## 📚 目录
1. 环境配置与初始化
2. 核心组件介绍
3. 工具函数说明
4. 单轮检索示例
5. 多轮自主检索示例

### 1. 环境配置与初始化

首先导入必要的库并配置 FlashRAG 框架。

In [1]:
from flashrag.utils import get_retriever, get_generator
from flashrag.config import Config
import re
from typing import List, Dict

/miniconda/envs/flashrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 1.1 配置参数说明

| 参数 | 说明 |
|------|------|
| `retrieval_method` | 检索方法，这里使用 e5 模型 |
| `model2path` | 模型路径映射 |
| `corpus_path` | 知识库语料路径 |
| `index_path` | FAISS 索引路径 |
| `retrieval_topk` | 每次检索返回的文档数量 |
| `generator_model_path` | 生成模型路径 |
| `faiss_gpu` | 是否使用 GPU 进行检索 |

FlashRAG支持更多参数调节，具体参考官方文档。

In [2]:
config_dict = {
    "retrieval_method": "e5",
    "model2path": {
        "e5": "/public/huggingface-models/intfloat/e5-base-v2",
    },
    "data_dir": "/root/FlashRAG/examples/quick_start/dataset/",
    "gpu_id": "0",
    "corpus_path": "/root/FlashRAG/examples/quick_start/indexes/general_knowledge.jsonl",
    "index_path": "/root/FlashRAG/examples/quick_start/indexes/e5_Flat.index",
    "faiss_gpu": False,
    "retrieval_topk": 5,
    "generator_model_path": "/public/huggingface-models/Qwen/QwQ-32B",
    "gpu_memory_utilization": 0.9,
}

# 创建配置对象
config = Config("/root/FlashRAG/examples/methods/my_config.yaml", config_dict)

#### 1.2 初始化检索器和生成器

- **检索器 (Retriever)**: 负责从知识库中检索相关文档
- **生成器 (Generator)**: 负责基于检索到的文档生成回答

说明：demo中使用的检索器为了快速演示采用的是缩减版的语料库，实际需要使用完整版会由本平台提供，无需手动下载

In [3]:
print("正在初始化检索器和生成器...")
retriever = get_retriever(config)
generator = get_generator(config)
print("✓ 初始化完成！")

正在初始化检索器和生成器...
INFO 11-07 11:56:58 [__init__.py:216] Automatically detected platform cuda.
INFO 11-07 11:56:58 [utils.py:233] non-default args: {'max_model_len': 2048, 'max_logprobs': 32016, 'disable_log_stats': True, 'model': '/public/huggingface-models/Qwen/QwQ-32B'}
INFO 11-07 11:56:58 [model.py:547] Resolved architecture: Qwen2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-07 11:56:58 [model.py:1510] Using max model len 2048


2025-11-07 11:56:59,094	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-07 11:56:59 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=16384.
WARNING 11-07 11:56:59 [__init__.py:3036] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 11-07 11:57:03 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:04 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:04 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/public/huggingface-models/Qwen/QwQ-32B', speculative_config=None, tokenizer='/public/huggingface-models/Qwen/QwQ-32B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_par

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/14 [00:01<00:23,  1.81s/it]
Loading safetensors checkpoint shards:  14% Completed | 2/14 [00:03<00:22,  1.84s/it]
Loading safetensors checkpoint shards:  21% Completed | 3/14 [00:05<00:20,  1.85s/it]
Loading safetensors checkpoint shards:  29% Completed | 4/14 [00:07<00:18,  1.85s/it]
Loading safetensors checkpoint shards:  36% Completed | 5/14 [00:09<00:16,  1.85s/it]
Loading safetensors checkpoint shards:  43% Completed | 6/14 [00:10<00:12,  1.51s/it]
Loading safetensors checkpoint shards:  50% Completed | 7/14 [00:11<00:11,  1.61s/it]
Loading safetensors checkpoint shards:  57% Completed | 8/14 [00:13<00:10,  1.69s/it]
Loading safetensors checkpoint shards:  64% Completed | 9/14 [00:15<00:08,  1.75s/it]
Loading safetensors checkpoint shards:  71% Completed | 10/14 [00:17<00:07,  1.78s/it]
Loading safetensors checkpoint shards:  79% Completed | 11/14

(EngineCore_DP0 pid=3676) INFO 11-07 11:57:31 [default_loader.py:267] Loading weights took 24.97 seconds
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:31 [gpu_model_runner.py:2653] Model loading took 61.0609 GiB and 25.236577 seconds
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:39 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/1aeca38780/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:39 [backends.py:559] Dynamo bytecode transform time: 7.80 s
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:42 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.596 s
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:44 [monitor.py:34] torch.compile takes 7.80 s in total
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:46 [gpu_worker.py:298] Available KV cache memory: 4.35 GiB
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:46 [kv_cache_utils.py:1087] GPU KV cache size: 17,792 tokens
(EngineCore_DP0 pid=3676) INFO 11-07

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:05<00:00, 12.53it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 67/67 [00:04<00:00, 15.95it/s]


(EngineCore_DP0 pid=3676) INFO 11-07 11:57:56 [gpu_model_runner.py:3480] Graph capturing finished in 10 secs, took 1.26 GiB
(EngineCore_DP0 pid=3676) INFO 11-07 11:57:56 [core.py:210] init engine (profile, create kv cache, warmup model) took 24.79 seconds
INFO 11-07 11:57:57 [llm.py:306] Supported_tasks: ['generate']
✓ 初始化完成！


### 2. 核心工具函数

以下是支持多轮检索问答系统的关键工具函数。

#### 2.1 文本提取函数

从生成的回复中提取特定标签之间的内容（如搜索查询、答案等）。

In [4]:
def extract_between(text: str, start_tag: str, end_tag: str):
    """
    提取文本中指定标签之间的内容
    
    参数:
        text: 原始文本
        start_tag: 起始标签
        end_tag: 结束标签
    
    返回:
        提取的内容（如果找到），否则返回 None
    
    示例:
        >>> text = "开始<query>人工智能</query>结束"
        >>> extract_between(text, "<query>", "</query>")
        '人工智能'
    """
    pattern = re.escape(start_tag) + r"(.*?)" + re.escape(end_tag)
    matches = re.findall(pattern, text, flags=re.DOTALL)
    if matches:
        return matches[-1].strip()  # 返回最后一个匹配项
    return None

In [5]:
# 测试提取函数
test_text = "这是一些文本 <|begin_search_query|>美国总统是谁<|end_search_query|> 更多文本"
extracted = extract_between(test_text, "<|begin_search_query|>", "<|end_search_query|>")
print(f"提取结果: {extracted}")

提取结果: 美国总统是谁


#### 2.2 文档格式化函数

将检索到的文档列表格式化为结构化字符串，方便模型理解。

In [6]:
def retrieved_docs_to_string(begin_doc_tag: str, end_doc_tag: str, retrieved_docs: List[Dict]):
    """
    将检索到的文档格式化为字符串
    
    参数:
        begin_doc_tag: 文档列表起始标签
        end_doc_tag: 文档列表结束标签
        retrieved_docs: 检索到的文档列表
    
    返回:
        格式化后的文档字符串
    
    格式示例:
        <|begin_search_result|>
        (1)Title: 文档标题1 Text: 文档内容1
        (2)Title: 文档标题2 Text: 文档内容2
        <|end_search_result|>
    """
    format_doc_string = ""
    for idx, doc in enumerate(retrieved_docs):
        contents = doc['contents']
        # 分离标题和正文
        title = contents.split('\n')[0]
        text = '\n'.join(contents.split('\n')[1:])
        doc_string = f"Title: {title} Text: {text}"
        # 移除开头的数字编号（如果有）
        doc_string = re.sub(r'^\d+\s+', '', doc_string)
        format_doc_string += f'({idx+1}){doc_string}\n'
    
    # 添加起始和结束标签
    format_doc_string = f'\n\n{begin_doc_tag}\n{format_doc_string}\n{end_doc_tag}\n\n'
    return format_doc_string

#### 2.3 系统提示词生成函数

生成指导模型进行多轮检索的系统提示词。

In [7]:
def get_multiqa_search_o1_instruction(MAX_SEARCH_LIMIT):
    """
    生成多轮检索问答系统的指令提示词
    
    参数:
        MAX_SEARCH_LIMIT: 最大搜索次数限制
    
    返回:
        系统提示词字符串
    """
    return (
        "You are a reasoning assistant with the ability to perform web searches to help "
        "you answer the user's question accurately. You have special tools:\n\n"
        "- To perform a search: write <|begin_search_query|> your query here <|end_search_query|>.\n"
        "Then, the system will search and analyze relevant web pages, then provide you with helpful information in the format <|begin_search_result|> ...search results... <|end_search_result|>.\n"
        "When you have gotten enough information to answer the user's question, stop searching and continue your reasoning to provide the your answer with <|begin_answer|> ... your answer... <|end_answer|>.\n\n"
        f"You can repeat the search process multiple times if necessary. The maximum number of search attempts is limited to {MAX_SEARCH_LIMIT}.\n\n"
        "Once you have all the information you need, continue your reasoning.\n\n"
        "Example:\n"
        "Question: \"Alice David is the voice of Lara Croft in a video game developed by which company?\"\n"
        "Assistant thinking steps:\n"
        "- I need to find out who voices Lara Croft in the video game.\n"
        "- Then, I need to determine which company developed that video game.\n\n"
        "Assistant:\n"
        "<|begin_search_query|>Alice David Lara Croft voice<|end_search_query|>\n\n"
        "(System returns processed information from relevant web pages)\n\n"
        "Assistant thinks: The search results indicate that Alice David is the voice of Lara Croft in a specific video game. Now, I need to find out which company developed that game.\n\n"
        "Assistant:\n"
        "<|begin_search_query|>video game developed by Alice David Lara Croft<|end_search_query|>\n\n"
        "(System returns processed information from relevant web pages)\n\n"
        "Assistant continues reasoning with the new information...\n\n"
        "Remember:\n"
        "- Use <|begin_search_query|> to request a web search and end with <|end_search_query|>.\n"
        "- When done searching, continue your reasoning.\n\n"
    )

In [8]:
# 配置参数
MAX_SEARCH_LIMIT = 3  # 最大搜索次数
stop_tokens = ["<|end_search_query|>", "<|end_answer|>"]  # 停止生成的标记

# 生成系统提示词
sysprompt = get_multiqa_search_o1_instruction(MAX_SEARCH_LIMIT)
print("系统提示词已生成")
print(f"前100个字符: {sysprompt[:100]}...")

系统提示词已生成
前100个字符: You are a reasoning assistant with the ability to perform web searches to help you answer the user's...


### 3. 基础功能演示

#### 3.1 批量检索示例

演示如何批量检索多个问题。

In [9]:
test_questions = [
    "Who is the president of the United States?",
    "What is the capital of France?",
    "Explain the theory of relativity.",
]

print("批量检索测试问题的相关文档...")
docs = retriever.batch_search(test_questions, 3)

for i, question in enumerate(test_questions):
    print(f"\n问题 {i+1}: {question}")
    print(f"检索到 {len(docs[i])} 个相关文档")
    if docs[i]:
        print(f"第一个文档标题: {docs[i][0]['contents'].split(chr(10))[0][:50]}...")

批量检索测试问题的相关文档...


Encoding process:   0%|          | 0/1 [00:00<?, ?it/s]

Use `query: ` as retreival instruction


Encoding process: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]


问题 1: Who is the president of the United States?
检索到 3 个相关文档
第一个文档标题: What is the name of the current president of the U...

问题 2: What is the capital of France?
检索到 3 个相关文档
第一个文档标题: What is the capital of France??...

问题 3: Explain the theory of relativity.
检索到 3 个相关文档
第一个文档标题: "Research Einsteins theory of relativity and provi...


#### 3.2 简单问答（无检索）

首先测试不使用检索的直接问答。

In [10]:
print("\n" + "="*50)
print("简单问答测试（不使用检索）")
print("="*50 + "\n")


for i, question in enumerate(test_questions):
    print(f"问题 {i+1}: {question}\n")
    prompt = f"Answer the following question. Question: {question}\nAssistant thinking steps:\n"
    
    # 生成回答
    response = generator.generate(
        prompt,
        max_new_tokens=512,
        temperature=0.1,
        stop=stop_tokens,
    )
    
    print(f"回答:\n{response[0]}\n")
    print("-"*50 + "\n")


简单问答测试（不使用检索）

问题 1: Who is the president of the United States?



Processed prompts: 100%|██████████| 1/1 [00:10<00:00, 10.05s/it, est. speed input: 1.99 toks/s, output: 40.00 toks/s]


回答:
Okay, so I need to figure out who the current president of the United States is. Let me start by recalling that the president serves a four-year term and can be re-elected. The last election I remember was in 2020, where Joe Biden was elected. He took office in January 2021. Before that, Donald Trump was the president from 2017 to 2021. Since the 2024 election hasn't happened yet, I think Biden is still in his first term. Wait, but when is the next election? It's usually in November, so the 2024 election would be next, but the current president as of now, in 2023, should still be Joe Biden. Let me double-check the dates. Biden's inauguration was January 20, 2021, so his term runs until January 20, 2025. Unless there's been some unexpected event, like resignation or impeachment, which I don't recall happening. So the answer should be Joe Biden. But maybe I should confirm the current date. If today is 2023, then yes, he's still president. If it's after January 20, 2025, that would be

Processed prompts: 100%|██████████| 1/1 [00:11<00:00, 11.08s/it, est. speed input: 1.63 toks/s, output: 42.08 toks/s]


回答:
Okay, so I need to figure out the capital of France. Let me start by recalling what I know about France. France is a country in Western Europe, right? I remember learning about some major cities there. Paris comes to mind immediately. Wait, isn't Paris the capital? But maybe I should double-check to be sure. Sometimes people confuse capitals with other major cities. For example, people might think Lyon or Marseille could be capitals because they're big cities, but I think Paris is definitely the capital. Let me think of other sources. In movies and books, Paris is often referred to as the capital. Also, the Eiffel Tower is in Paris, which is a symbol of France. Plus, in school, when we studied European capitals, France's was listed as Paris. I can't recall any other city being mentioned for that role. Maybe I can think of government institutions. The President of France has offices in Paris, like the Élysée Palace. The French Parliament is there too. That solidifies it as the capit

Processed prompts: 100%|██████████| 1/1 [00:12<00:00, 12.15s/it, est. speed input: 1.48 toks/s, output: 42.14 toks/s]

回答:
Okay, so I need to explain the theory of relativity. Hmm, where do I start? I remember that there are two parts: special and general relativity. Let me think. Special relativity was introduced first by Einstein in 1905, right? It deals with objects moving at constant speed, especially near the speed of light. Oh, and the famous equation E=mc² comes from that. But wait, what's the main idea here? Maybe something about the laws of physics being the same for all non-accelerating observers, and the speed of light is constant regardless of the observer's motion. That sounds familiar.

Then there's general relativity, which came later, around 1915. That one's about gravity, right? Instead of gravity as a force, Einstein described it as the curvature of spacetime caused by mass and energy. So massive objects like planets warp the spacetime around them, and other objects move along that curvature, which we perceive as gravity. Like how a ball on a stretched sheet would make a dip, and othe

### 4. 单轮检索完整流程

演示一个完整的单轮检索问答流程：
1. 模型生成搜索查询
2. 检索相关文档
3. 基于文档生成最终答案

In [11]:
print("\n" + "="*50)
print("单轮检索完整流程示例")
print("="*50 + "\n")

# 选择第一个问题进行演示
question = test_questions[0]
print(f"📝 问题: {question}\n")

# 步骤1: 生成搜索查询
print("步骤1: 让模型生成搜索查询...")
prompt = sysprompt + f"Question: {question}\n"
response = generator.generate(
    prompt, 
    max_new_tokens=512, 
    temperature=0.1, 
    stop=stop_tokens
)[0]
print(f"模型回复:\n{response}\n")

# 步骤2: 提取搜索查询
search_query = extract_between(response, "<|begin_search_query|>", "<|end_search_query|>")
print(f"🔍 提取的搜索查询: {search_query}\n")

if search_query:
    # 步骤3: 执行检索
    print("步骤2: 执行检索...")
    docs = retriever.search(search_query, 3)
    print(f"检索到 {len(docs)} 个相关文档\n")
    
    # 步骤4: 格式化文档
    doc_str = retrieved_docs_to_string("<|begin_search_result|>", "<|end_search_result|>", docs)
    print(f"格式化文档（前200字符）:\n{doc_str[:200]}...\n")
    
    # 步骤5: 基于检索结果生成最终答案
    print("步骤3: 基于检索结果生成最终答案...")
    prompt = prompt + response + doc_str + '\n'
    response = generator.generate(
        prompt, 
        max_new_tokens=512, 
        temperature=0.1, 
        stop=stop_tokens
    )[0]
    print(f"模型回复:\n{response}\n")
    
    # 步骤6: 提取最终答案
    answer = extract_between(response, "<|begin_answer|>", "<|end_answer|>")
    if answer:
        print(f"✅ 最终答案:\n{answer}\n")
    else:
        print("⚠️ 未找到明确的答案标记\n")
else:
    print("⚠️ 模型未生成搜索查询\n")


单轮检索完整流程示例

📝 问题: Who is the president of the United States?

步骤1: 让模型生成搜索查询...


Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.06s/it, est. speed input: 121.47 toks/s, output: 41.91 toks/s]


模型回复:
Alright, the user is asking who the current president of the United States is. I know that this information can change over time, so I need to make sure I have the most up-to-date answer. Let me start by recalling that as of my last update in July 2024, the president was Joe Biden. But to be certain, I should perform a quick search to confirm.

First, I'll search for the current president of the United States. The search results should provide the latest information. Let me check that.

<|begin_search_query|>current president of the United States<|end_search_query|>

🔍 提取的搜索查询: current president of the United States

步骤2: 执行检索...


Encoding process: 100%|██████████| 1/1 [00:00<00:00, 27.85it/s]


检索到 3 个相关文档

格式化文档（前200字符）:


<|begin_search_result|>
(1)Title: Who is the current President of the United States? Text: The current President of the United States is Joe Biden.\n
(2)Title: What is the name of the current presid...

步骤3: 基于检索结果生成最终答案...


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.88s/it, est. speed input: 320.69 toks/s, output: 42.08 toks/s]

模型回复:
Okay, all the search results confirm that the current president is Joe Biden. There's no conflicting information here. Since the user's question is straightforward and the search results are consistent, I can confidently provide the answer without needing further searches. I should make sure to state the answer clearly.
<|begin_answer|>
The current president of the United States is Joe Biden. <|end_answer|>

✅ 最终答案:
The current president of the United States is Joe Biden.



### 5. 多轮自主检索系统

实现一个完整的多轮检索系统，模型可以自主决定是否需要继续搜索。

In [12]:
def multi_turn_qa(question: str, max_turns: int = 3, verbose: bool = True):
    """
    多轮检索问答函数
    
    参数:
        question: 用户问题
        max_turns: 最大检索轮数
        verbose: 是否打印详细信息
    
    返回:
        final_answer: 最终答案
        search_history: 搜索历史记录
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"开始多轮检索问答")
        print(f"{'='*60}\n")
        print(f"📝 问题: {question}\n")
    
    # 初始化
    sysprompt = get_multiqa_search_o1_instruction(max_turns)
    prompt = sysprompt + f"Question: {question}\n"
    search_history = []
    
    for turn in range(max_turns):
        if verbose:
            print(f"\n--- 第 {turn + 1} 轮 ---\n")
        
        # 生成回复
        response = generator.generate(
            prompt,
            max_new_tokens=512,
            temperature=0.1,
            stop=stop_tokens
        )[0]
        
        if verbose:
            print(f"模型回复:\n{response}\n")
        
        # 检查是否生成了搜索查询
        search_query = extract_between(response, "<|begin_search_query|>", "<|end_search_query|>")
        
        if search_query:
            if verbose:
                print(f"🔍 检测到搜索查询: {search_query}")
            
            # 执行检索
            docs = retriever.search(search_query, 3)
            search_history.append({
                'turn': turn + 1,
                'query': search_query,
                'num_docs': len(docs)
            })
            
            if verbose:
                print(f"✓ 检索到 {len(docs)} 个文档\n")
            
            # 格式化文档并添加到提示词
            doc_str = retrieved_docs_to_string("<|begin_search_result|>", "<|end_search_result|>", docs)
            prompt = prompt + response + doc_str + '\n'
            
        else:
            # 检查是否给出了最终答案
            answer = extract_between(response, "<|begin_answer|>", "<|end_answer|>")
            
            if answer:
                if verbose:
                    print(f"✅ 找到最终答案！\n")
                return answer, search_history
            else:
                if verbose:
                    print("⚠️ 未检测到搜索查询或最终答案，继续...\n")
                prompt = prompt + response + '\n'
    
    if verbose:
        print(f"\n⚠️ 达到最大检索轮数 ({max_turns})，尝试提取答案...\n")
    
    # 达到最大轮数，尝试提取答案
    final_response = generator.generate(
        prompt + "Please provide your final answer with <|begin_answer|> ... <|end_answer|>.\n",
        max_new_tokens=512,
        temperature=0.1,
        stop=stop_tokens
    )[0]
    
    answer = extract_between(final_response, "<|begin_answer|>", "<|end_answer|>")
    return answer if answer else "未能生成明确答案", search_history

#### 5.1 测试多轮检索系统

In [13]:
# 测试问题
complex_question = "Who is the president of the United States?"

# 执行多轮检索问答
final_answer, search_history = multi_turn_qa(
    complex_question, 
    max_turns=3, 
    verbose=True
)

# 输出结果摘要
print("\n" + "="*60)
print("结果摘要")
print("="*60 + "\n")
print(f"问题: {complex_question}\n")
print(f"搜索轮数: {len(search_history)}")
for search in search_history:
    print(f"  - 第{search['turn']}轮: 查询=\"{search['query']}\", 文档数={search['num_docs']}")
print(f"\n最终答案:\n{final_answer}\n")


开始多轮检索问答

📝 问题: Who is the president of the United States?


--- 第 1 轮 ---



Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.04s/it, est. speed input: 122.10 toks/s, output: 42.13 toks/s]


模型回复:
Alright, the user is asking who the current president of the United States is. I know that this information can change over time, so I need to make sure I have the most up-to-date answer. Let me start by recalling that as of my last update in July 2024, the president was Joe Biden. But to be certain, I should perform a quick search to confirm.

First, I'll search for the current president of the United States. The search results should provide the latest information. Let me check that.

<|begin_search_query|>current president of the United States<|end_search_query|>

🔍 检测到搜索查询: current president of the United States


Encoding process: 100%|██████████| 1/1 [00:00<00:00, 57.03it/s]


✓ 检索到 3 个文档


--- 第 2 轮 ---



Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.88s/it, est. speed input: 320.88 toks/s, output: 42.11 toks/s]

模型回复:
Okay, all the search results confirm that the current president is Joe Biden. There's no conflicting information here. Since the user's question is straightforward and the search results are consistent, I can confidently provide the answer without needing further searches. I should make sure to state the answer clearly.
<|begin_answer|>
The current president of the United States is Joe Biden. <|end_answer|>

✅ 找到最终答案！


结果摘要

问题: Who is the president of the United States?

搜索轮数: 1
  - 第1轮: 查询="current president of the United States", 文档数=3

最终答案:
The current president of the United States is Joe Biden.

